# Анализ графа друзей VK

**Цель работы:** по ссылке, короткому имени или числовому ID пользователя VK собрать доступную сеть друзей, построить и отобразить граф, оценить центральность его вершин и найти пересечение друзей для двух или трёх страниц.

В работе рассчитываются:
- **центральность по посредничеству** (`betweenness`) — как часто вершина лежит на кратчайших путях между другими вершинами;
- **центральность по близости** (`closeness`) — насколько близко вершина расположена ко всем остальным;
- **центральность собственного вектора** (`eigenvector`) — насколько вершина связана с другими важными вершинами;
- дополнительно — степень вершины и нормированная центральность по степени.

> VK возвращает только сведения, разрешённые настройками приватности и правами токена. Закрытые или удалённые страницы могут отсутствовать. Полученные идентификаторы относятся к персональным данным: используйте их только в учебных целях, не публикуйте токен и результаты без согласия участников.

## 1. Подготовка окружения

Если при импорте возникает `ModuleNotFoundError`, раскомментируйте следующую строку и выполните ячейку один раз.

In [ ]:
# %pip install requests networkx pandas matplotlib

import getpass
import json
import math
import time
from itertools import combinations
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests

plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_colwidth', 100)

## 2. Клиент VK API

Нужен пользовательский access token VK API с доступом к чтению друзей. Токен вводится через скрытое поле и существует только в памяти текущего ядра — в ноутбук он не записывается. Во всех сетевых запросах установлен тайм-аут.

Метод `execute` объединяет до 25 запросов списков друзей в один вызов API. Это заметно ускоряет построение связей между друзьями и уменьшает вероятность превысить лимит запросов.

In [ ]:
VK_API_VERSION = '5.199'
VK_API_URL = 'https://api.vk.com/method'
REQUEST_TIMEOUT_SECONDS = 15
PAUSE_BETWEEN_BATCHES_SECONDS = 0.35


class VKAPIError(RuntimeError):
    """Безопасная ошибка VK API без вывода токена и параметров запроса."""


def normalize_vk_identifier(value):
    """Преобразует URL VK, @domain, id123 или число в значение для users.get."""
    text = str(value).strip()
    if not text:
        raise ValueError('Пустой идентификатор пользователя')

    lowered = text.lower()
    looks_like_vk_url = (
        '://' in text
        or lowered.startswith(('vk.com/', 'vk.ru/', 'www.vk.com/', 'www.vk.ru/', 'm.vk.com/', 'm.vk.ru/'))
    )
    if looks_like_vk_url:
        url = text if '://' in text else f'https://{text}'
        parsed = urlparse(url)
        hostname = (parsed.hostname or '').lower()
        if hostname.removeprefix('www.').removeprefix('m.') not in {'vk.com', 'vk.ru'}:
            raise ValueError(f'Ожидалась ссылка на vk.com или vk.ru: {value}')
        path_parts = [part for part in parsed.path.split('/') if part]
        if not path_parts:
            raise ValueError(f'В ссылке не найден пользователь: {value}')
        text = path_parts[0]

    return text.lstrip('@').split('?')[0]


def profile_name(profile):
    return f"{profile.get('first_name', '')} {profile.get('last_name', '')}".strip() or str(profile['id'])


def profile_url(profile):
    domain = profile.get('screen_name') or profile.get('domain')
    return f"https://vk.com/{domain or ('id' + str(profile['id']))}"


class VKClient:
    def __init__(self, token, api_version=VK_API_VERSION, timeout=REQUEST_TIMEOUT_SECONDS):
        if not token:
            raise ValueError('Токен VK API не задан')
        self._token = token
        self._api_version = api_version
        self._timeout = timeout
        self._session = requests.Session()

    def call(self, method, **params):
        request_data = {
            **params,
            'access_token': self._token,
            'v': self._api_version,
        }
        try:
            response = self._session.post(
                f'{VK_API_URL}/{method}',
                data=request_data,
                timeout=self._timeout,
            )
        except requests.RequestException as error:
            raise VKAPIError(f'Сетевая ошибка при вызове {method}: {type(error).__name__}') from error

        if not response.ok:
            raise VKAPIError(f'VK API вернул HTTP {response.status_code} для метода {method}')

        try:
            payload = response.json()
        except requests.JSONDecodeError as error:
            raise VKAPIError(f'VK API вернул некорректный JSON для метода {method}') from error

        if 'error' in payload:
            api_error = payload['error']
            code = api_error.get('error_code', 'unknown')
            message = api_error.get('error_msg', 'неизвестная ошибка')
            raise VKAPIError(f'Ошибка VK API {code}: {message}')

        return payload.get('response')

    def resolve_user(self, value):
        identifier = normalize_vk_identifier(value)
        users = self.call(
            'users.get',
            user_ids=identifier,
            fields='screen_name,domain,photo_100',
        )
        if not users:
            raise VKAPIError(f'Пользователь не найден: {value}')
        return users[0]

    def get_users(self, user_ids):
        ids = list(dict.fromkeys(int(user_id) for user_id in user_ids))
        users = []
        for start in range(0, len(ids), 1000):
            chunk = ids[start:start + 1000]
            users.extend(self.call(
                'users.get',
                user_ids=','.join(map(str, chunk)),
                fields='screen_name,domain,photo_100',
            ))
            time.sleep(PAUSE_BETWEEN_BATCHES_SECONDS)
        return users

    def get_friends(self, user_id, limit=None, with_profiles=False):
        result_items = []
        offset = 0
        fields = 'screen_name,domain,photo_100' if with_profiles else None

        while limit is None or len(result_items) < limit:
            count = 5000 if limit is None else min(5000, limit - len(result_items))
            params = {'user_id': int(user_id), 'count': count, 'offset': offset}
            if fields:
                params['fields'] = fields
            page = self.call('friends.get', **params)
            items = page.get('items', [])
            result_items.extend(items)
            offset += len(items)

            if not items or offset >= page.get('count', 0):
                break
            time.sleep(PAUSE_BETWEEN_BATCHES_SECONDS)

        return result_items[:limit] if limit is not None else result_items

    def get_friend_id_sets(self, user_ids):
        """Возвращает доступные списки друзей сразу для нескольких пользователей."""
        ids = [int(user_id) for user_id in user_ids]
        friend_sets = {}
        unavailable = []

        for start in range(0, len(ids), 25):
            chunk = ids[start:start + 25]
            code = (
                f'var ids={json.dumps(chunk)};'
                'var out=[]; var i=0;'
                'while(i<ids.length){'
                'out.push(API.friends.get({"user_id":ids[i],"count":5000}));'
                'i=i+1;'
                '} return out;'
            )
            batch = self.call('execute', code=code)
            batch = batch if isinstance(batch, list) else []

            for index, user_id in enumerate(chunk):
                item = batch[index] if index < len(batch) else False
                if not isinstance(item, dict):
                    unavailable.append(user_id)
                    continue
                friend_sets[user_id] = set(item.get('items', []))

            time.sleep(PAUSE_BETWEEN_BATCHES_SECONDS)

        return friend_sets, unavailable

In [ ]:
VK_TOKEN = getpass.getpass('Вставьте access token VK (ввод скрыт): ').strip()
vk = VKClient(VK_TOKEN)
print('Клиент VK API создан; токен в ноутбук не сохранён.')

## 3. Построение графа пользователя

Вершины графа — выбранный пользователь и его доступные друзья. Рёбра соединяют пользователя со всеми выбранными друзьями, а также друзей, которые присутствуют в доступных списках друзей друг друга.

Параметр `max_friends` ограничивает размер графа и время загрузки. Значение `None` означает всех доступных друзей. Если задан предел, результат описывает только выбранную API подвыборку, а не всю социальную сеть.

In [ ]:
def build_friendship_graph(client, person, max_friends=150):
    owner = client.resolve_user(person)
    friends = client.get_friends(owner['id'], limit=max_friends, with_profiles=True)

    graph = nx.Graph()
    graph.add_node(
        owner['id'],
        name=profile_name(owner),
        url=profile_url(owner),
        role='owner',
    )

    friend_by_id = {int(friend['id']): friend for friend in friends}
    friend_ids = set(friend_by_id)

    for friend_id, friend in friend_by_id.items():
        graph.add_node(
            friend_id,
            name=profile_name(friend),
            url=profile_url(friend),
            role='friend',
        )
        graph.add_edge(owner['id'], friend_id)

    visible_lists, unavailable = client.get_friend_id_sets(sorted(friend_ids))
    for source_id, source_friends in visible_lists.items():
        for target_id in source_friends & friend_ids:
            if source_id < target_id:
                graph.add_edge(source_id, target_id)

    info = {
        'owner': owner,
        'selected_friends': len(friend_ids),
        'readable_friend_lists': len(visible_lists),
        'unavailable_friend_lists': len(unavailable),
        'max_friends': max_friends,
    }
    return graph, info


def graph_summary(graph, info):
    possible_edges = graph.number_of_nodes() * (graph.number_of_nodes() - 1) / 2
    density = graph.number_of_edges() / possible_edges if possible_edges else 0.0
    return pd.Series({
        'Пользователь': profile_name(info['owner']),
        'URL': profile_url(info['owner']),
        'Вершин': graph.number_of_nodes(),
        'Рёбер': graph.number_of_edges(),
        'Плотность': round(density, 4),
        'Доступно списков друзей': info['readable_friend_lists'],
        'Недоступно списков друзей': info['unavailable_friend_lists'],
    })

Укажите страницу в любом из форматов: `https://vk.com/domain`, `https://vk.ru/domain`, `domain`, `@domain`, `id123` или `123`. Для быстрой проверки рекомендуется начать со 100–150 друзей.

In [ ]:
PERSON = input('Страница пользователя VK: ').strip()
MAX_FRIENDS = 150  # поставьте None, чтобы взять всех доступных друзей

friend_graph, graph_info = build_friendship_graph(vk, PERSON, max_friends=MAX_FRIENDS)
graph_summary(friend_graph, graph_info).to_frame('Значение')

## 4. Расчёт центральностей

Показатели нормированы и лежат в диапазоне от 0 до 1. Центральность собственного вектора вычисляется итерационно. Центральная страница почти всегда получит высокие значения, потому что по построению соединена со всеми друзьями; поэтому в таблице удобно отдельно анализировать строки с `Это исходная страница = False`.

In [ ]:
def calculate_centralities(graph):
    if graph.number_of_nodes() == 0:
        return pd.DataFrame()

    betweenness = nx.betweenness_centrality(graph, normalized=True)
    closeness = nx.closeness_centrality(graph)

    if graph.number_of_nodes() == 1:
        eigenvector = {next(iter(graph.nodes)): 1.0}
    else:
        try:
            eigenvector = nx.eigenvector_centrality(
                graph, max_iter=2000, tol=1e-8
            )
        except nx.PowerIterationFailedConvergence:
            eigenvector = {node_id: np.nan for node_id in graph.nodes}
            print('Eigenvector centrality не сошлась; в столбце оставлены NaN.')

    degree_centrality = nx.degree_centrality(graph)
    rows = []
    for node_id, attributes in graph.nodes(data=True):
        rows.append({
            'VK ID': node_id,
            'Имя': attributes['name'],
            'URL': attributes['url'],
            'Это исходная страница': attributes['role'] == 'owner',
            'Степень': graph.degree(node_id),
            'По степени': degree_centrality[node_id],
            'По посредничеству': betweenness[node_id],
            'По близости': closeness[node_id],
            'Собственного вектора': eigenvector[node_id],
        })

    return (
        pd.DataFrame(rows)
        .set_index('VK ID')
        .sort_values(
            ['Собственного вектора', 'По близости', 'По посредничеству'],
            ascending=False,
        )
    )


centrality_table = calculate_centralities(friend_graph)
display(centrality_table.head(15).round(5))

print('Топ-10 друзей без исходной страницы:')
display(
    centrality_table.loc[~centrality_table['Это исходная страница']]
    .head(10)
    .round(5)
)

### Сравнение лидеров по разным мерам

Один и тот же человек может быть важен по разным причинам: связывать сообщества, находиться близко к большинству или быть связанным с другими центральными людьми.

In [ ]:
def plot_centrality_leaders(table, top_n=10):
    metrics = ['По посредничеству', 'По близости', 'Собственного вектора']
    titles = ['Посредничество', 'Близость', 'Собственный вектор']
    data = table.loc[~table['Это исходная страница']]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for axis, metric, title in zip(axes, metrics, titles):
        leaders = data.nlargest(min(top_n, len(data)), metric).sort_values(metric)
        axis.barh(leaders['Имя'], leaders[metric], color='#4472C4')
        axis.set_title(f'Топ-{len(leaders)}: {title}')
        axis.set_xlabel('Центральность')
        axis.grid(axis='x', alpha=0.25)

    plt.tight_layout()
    plt.show()


plot_centrality_leaders(centrality_table)

## 5. Визуализация графа

Размер вершины соответствует центральности собственного вектора. Красным отмечена исходная страница, жёлтым — десять наиболее центральных вершин, синим — остальные. Подписи оставлены только для исходной страницы и лидеров, чтобы граф не превращался в нечитаемое облако текста.

In [ ]:
def plot_friendship_graph(graph, centralities, labels_count=10):
    if graph.number_of_nodes() == 0:
        print('Граф пуст.')
        return

    owner_ids = {
        node_id for node_id, attrs in graph.nodes(data=True)
        if attrs['role'] == 'owner'
    }
    leader_ids = set(centralities.head(labels_count).index) | owner_ids
    node_colors = [
        '#D62728' if node_id in owner_ids else
        '#F2C14E' if node_id in leader_ids else '#69A7D3'
        for node_id in graph.nodes
    ]
    eigenvector_values = centralities['Собственного вектора'].fillna(0).to_dict()
    node_sizes = [500 + 5000 * eigenvector_values[node_id] for node_id in graph.nodes]

    node_count = graph.number_of_nodes()
    layout_k = max(0.12, 2 / math.sqrt(node_count))
    positions = nx.spring_layout(graph, seed=42, k=layout_k, iterations=100)
    labels = {node_id: graph.nodes[node_id]['name'] for node_id in leader_ids}

    plt.figure(figsize=(15, 11))
    nx.draw_networkx_edges(graph, positions, alpha=0.16, width=0.7, edge_color='#555555')
    nx.draw_networkx_nodes(
        graph, positions, node_color=node_colors, node_size=node_sizes,
        alpha=0.9, linewidths=0.6, edgecolors='white'
    )
    nx.draw_networkx_labels(graph, positions, labels=labels, font_size=8)
    plt.title(
        f"Граф друзей: {graph.number_of_nodes()} вершин, "
        f"{graph.number_of_edges()} рёбер"
    )
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('vk_friend_graph.png', dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()


plot_friendship_graph(friend_graph, centrality_table)

## 6. Пересечение друзей двух или трёх страниц

Функция принимает ровно 2 или 3 значения в тех же форматах, что и ранее. Для трёх страниц выводятся размеры всех попарных пересечений и общего пересечения трёх множеств. В граф входят друзья, встречающиеся хотя бы у двух переданных страниц, и доступные связи между ними. Зелёным отмечается строгое пересечение всех страниц, жёлтым — друзья, общие ровно для двух из трёх страниц. Поэтому граф будет построен даже при пустом тройном пересечении; если нет даже попарных совпадений, на нём останутся три исходные страницы.

In [ ]:
def find_common_friends(client, people, max_friends=None):
    if len(people) not in (2, 3):
        raise ValueError('Нужно передать ровно 2 или 3 страницы VK')

    profiles = [client.resolve_user(person) for person in people]
    profile_ids = [profile['id'] for profile in profiles]
    if len(set(profile_ids)) != len(profile_ids):
        raise ValueError('Переданы повторяющиеся страницы')

    friend_sets = []
    for profile in profiles:
        friend_ids = client.get_friends(profile['id'], limit=max_friends)
        friend_sets.append(set(map(int, friend_ids)))

    summary_rows = []
    for size in range(1, len(profiles) + 1):
        for indices in combinations(range(len(profiles)), size):
            names = [profile_name(profiles[index]) for index in indices]
            intersection = set.intersection(*(friend_sets[index] for index in indices))
            summary_rows.append({
                'Множество': ' ∩ '.join(names),
                'Количество': len(intersection),
            })

    membership_by_id = {}
    for profile_index, friend_set in enumerate(friend_sets):
        for friend_id in friend_set:
            membership_by_id.setdefault(friend_id, set()).add(profile_index)

    common_ids = set.intersection(*friend_sets)
    shared_ids = {
        friend_id for friend_id, memberships in membership_by_id.items()
        if len(memberships) >= 2
    }
    shared_profiles = client.get_users(sorted(shared_ids)) if shared_ids else []
    profile_by_id = {int(profile['id']): profile for profile in shared_profiles}

    def make_table(friend_ids):
        rows = []
        for friend_id in sorted(friend_ids):
            profile = profile_by_id.get(friend_id)
            if profile is None:
                continue
            memberships = membership_by_id[friend_id]
            rows.append({
                'VK ID': friend_id,
                'Имя': profile_name(profile),
                'URL': profile_url(profile),
                'Общих страниц': len(memberships),
                'Страницы': ', '.join(profile_name(profiles[index]) for index in sorted(memberships)),
            })
        return pd.DataFrame(
            rows, columns=['VK ID', 'Имя', 'URL', 'Общих страниц', 'Страницы']
        ).sort_values(['Общих страниц', 'Имя'], ascending=[False, True]).reset_index(drop=True)

    common_table = make_table(common_ids)
    shared_table = make_table(shared_ids)

    return {
        'profiles': profiles,
        'friend_sets': friend_sets,
        'common_ids': common_ids,
        'shared_ids': shared_ids,
        'membership_by_id': membership_by_id,
        'common_table': common_table,
        'shared_table': shared_table,
        'summary': pd.DataFrame(summary_rows),
        'max_friends': max_friends,
    }


def build_common_friends_graph(client, result):
    """Строит граф друзей, общих хотя бы для двух переданных страниц."""
    profiles = result['profiles']
    shared_table = result['shared_table']
    common_ids = set(map(int, result['common_ids']))
    shared_ids = set(map(int, result['shared_ids']))
    membership_by_id = result['membership_by_id']
    graph = nx.Graph()

    owner_node_ids = []
    for profile in profiles:
        node_id = f"person_{profile['id']}"
        owner_node_ids.append(node_id)
        graph.add_node(
            node_id, name=profile_name(profile), url=profile_url(profile), kind='person'
        )

    for _, row in shared_table.iterrows():
        friend_id = int(row['VK ID'])
        friend_node_id = f'friend_{friend_id}'
        memberships = membership_by_id[friend_id]
        kind = 'all_common' if friend_id in common_ids else 'pairwise_common'
        graph.add_node(
            friend_node_id, name=row['Имя'], url=row['URL'], kind=kind,
            membership_count=len(memberships),
        )
        for profile_index in memberships:
            graph.add_edge(
                owner_node_ids[profile_index], friend_node_id, relation='common_friend'
            )

    visible_lists, unavailable = client.get_friend_id_sets(sorted(shared_ids))
    internal_edges = 0
    for source_id, source_friends in visible_lists.items():
        for target_id in source_friends & shared_ids:
            if source_id < target_id:
                graph.add_edge(
                    f'friend_{source_id}', f'friend_{target_id}', relation='friendship'
                )
                internal_edges += 1

    info = {
        'Передано страниц': len(profiles),
        'Общих минимум для двух страниц': len(shared_ids),
        'Общих для всех страниц': len(common_ids),
        'Связей между общими друзьями': internal_edges,
        'Доступно списков общих друзей': len(visible_lists),
        'Недоступно списков общих друзей': len(unavailable),
    }
    return graph, info


def plot_common_friends(graph, info, labels_count=25):
    common_nodes = [
        node_id for node_id, attrs in graph.nodes(data=True)
        if attrs['kind'] != 'person'
    ]
    if not common_nodes:
        print('Общих друзей даже для двух из указанных страниц не найдено; показаны сами страницы.')

    person_nodes = [
        node_id for node_id, attrs in graph.nodes(data=True)
        if attrs['kind'] == 'person'
    ]
    internal_edges = [
        (source, target) for source, target, attrs in graph.edges(data=True)
        if attrs['relation'] == 'friendship'
    ]
    person_edges = [
        (source, target) for source, target, attrs in graph.edges(data=True)
        if attrs['relation'] == 'common_friend'
    ]

    person_angles = np.linspace(np.pi / 2, np.pi / 2 + 2 * np.pi, len(person_nodes), endpoint=False)
    person_positions = {
        node_id: 0.65 * np.array([np.cos(angle), np.sin(angle)])
        for node_id, angle in zip(person_nodes, person_angles)
    }
    if common_nodes:
        node_count = graph.number_of_nodes()
        layout_k = max(0.12, 0.9 / math.sqrt(node_count))
        positions = nx.spring_layout(
            graph, seed=42, pos=person_positions, fixed=person_nodes,
            k=layout_k, iterations=200
        )
        positions = nx.rescale_layout_dict(positions, scale=0.85)
    else:
        positions = person_positions
    color_by_kind = {
        'person': '#D62728',
        'all_common': '#70AD47',
        'pairwise_common': '#F2C14E',
    }
    colors = [color_by_kind[graph.nodes[node_id]['kind']] for node_id in graph.nodes]
    sizes = [
        1900 if graph.nodes[node_id]['kind'] == 'person'
        else 450 + 35 * graph.degree(node_id)
        for node_id in graph.nodes
    ]

    labelled_common = sorted(common_nodes, key=graph.degree, reverse=True)[:labels_count]
    labelled_nodes = set(person_nodes) | set(labelled_common)
    labels = {node_id: graph.nodes[node_id]['name'] for node_id in labelled_nodes}

    plt.figure(figsize=(11, 8))
    nx.draw_networkx_edges(
        graph, positions, edgelist=person_edges, alpha=0.18, width=0.8, edge_color='#777777'
    )
    nx.draw_networkx_edges(
        graph, positions, edgelist=internal_edges, alpha=0.55, width=1.4, edge_color='#1B5E20'
    )
    nx.draw_networkx_nodes(
        graph, positions, node_color=colors, node_size=sizes,
        edgecolors='white', linewidths=0.8
    )
    nx.draw_networkx_labels(graph, positions, labels=labels, font_size=8)
    plt.title(
        f"Граф пересечений: {info['Общих минимум для двух страниц']} общих минимум для двух, "
        f"{info['Общих для всех страниц']} общих для всех; "
        f"{info['Связей между общими друзьями']} связей между ними"
    )
    plt.scatter([], [], s=180, c='#D62728', label='Переданные страницы')
    plt.scatter([], [], s=100, c='#70AD47', label='Общие для всех страниц')
    if info['Передано страниц'] == 3:
        plt.scatter([], [], s=100, c='#F2C14E', label='Общие ровно для двух страниц')
    plt.legend(loc='best')
    plt.axis('off')
    plt.margins(0.12)
    plt.tight_layout()
    plt.savefig('vk_common_friends.png', dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()

In [ ]:
raw_people = input(
    'Введите 2 или 3 страницы VK через запятую: '
).split(',')
PEOPLE_FOR_INTERSECTION = [person.strip() for person in raw_people if person.strip()]

intersection_result = find_common_friends(
    vk, PEOPLE_FOR_INTERSECTION, max_friends=None
)
display(intersection_result['summary'])

print('Друзья, общие для всех указанных страниц:')
display(intersection_result['common_table'])

if len(PEOPLE_FOR_INTERSECTION) == 3:
    print('Все друзья, общие хотя бы для двух страниц:')
    display(intersection_result['shared_table'])

common_graph, common_graph_info = build_common_friends_graph(vk, intersection_result)
display(pd.Series(common_graph_info).to_frame('Значение'))
plot_common_friends(common_graph, common_graph_info)

## 7. Сохранение результатов

Таблицы сохраняются в CSV, а основной граф и граф пересечения — в отдельных файлах GraphML. GraphML можно открыть в Gephi, Cytoscape или другом редакторе графов. Визуализации Matplotlib автоматически сохраняются как `vk_friend_graph.png` и `vk_common_friends.png` при выполнении соответствующих ячеек. Токен в эти файлы не попадает.

In [ ]:
centrality_table.to_csv('vk_centralities.csv', encoding='utf-8-sig')
nx.write_graphml(friend_graph, 'vk_friend_graph.graphml')
nx.write_graphml(common_graph, 'vk_common_friends_graph.graphml')
intersection_result['common_table'].to_csv(
    'vk_common_friends.csv', index=False, encoding='utf-8-sig'
)
print('Сохранены CSV и оба GraphML; PNG создаются ячейками визуализации Matplotlib.')

## 8. Интерпретация и ограничения

1. Высокое **посредничество** указывает на возможного связующего между группами друзей, но не доказывает реальное влияние человека.
2. Высокая **близость** означает короткие пути до остальных вершин именно в собранном графе.
3. Высокая центральность **собственного вектора** появляется у вершин, связанных с другими центральными вершинами.
4. Исходная страница искусственно соединена со всеми своими друзьями, поэтому её показатели закономерно завышены; для содержательного сравнения друзей она исключается из диаграмм лидеров.
5. Граф неполон: приватность, блокировки API и `MAX_FRIENDS` скрывают часть вершин и рёбер. Поэтому показатели характеризуют собранную выборку, а не всю сеть VK.
6. Совпадение в списках друзей означает связь в VK, но не обязательно близкое знакомство в реальной жизни.
7. Для воспроизводимого отчёта зафиксируйте дату сбора, значение `MAX_FRIENDS`, число недоступных списков и версию VK API.